# Step 3: Does position cleaning restore conflict detection?

Methods: `none` (step 2), `savgol` (Savitzky–Golay), `kalman` (constant-velocity Kalman smoother,
noise level estimated from the data), `denoiser_jitter` (learned, trained on Gaussian jitter only)
and `denoiser_mixed` (learned, noise-aware: trained on all position and detection noise types).
All methods only clean positions and fill missed frames; velocity is then derived exactly as in
the reference. Kind `clean` applies each method to noise-free positions.

Run the scripts from the Anaconda Prompt first (`05_train_denoisers.py`, then
`06_filter_experiments.py`); this notebook loads the results.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import Image, display
from sensors_noise.filter_pipeline import collect_filter_results

OUT_DIR = Path.cwd().parent / "results" / "03_filters"      # or "03_filters_quick"
res = collect_filter_results(OUT_DIR, verbose=False)

## Recall and precision, all road-user groups pooled

In [ ]:
d = res["detection_pooled"].reset_index()
for col in ("precision", "recall"):
    print(col)
    display(d.pivot_table(index=["kind", "level"], columns=["measure", "method"], values=col).round(2))

## By road-user group

In [ ]:
g = res["detection"].reset_index()
g.pivot_table(index=["kind", "level", "group"], columns=["measure", "method"], values="precision").round(2)

## TTC / PET value errors on conflicts found (median signed error, s)

In [ ]:
e = res["errors_pooled"].reset_index()
e.pivot_table(index=["kind", "level"], columns=["measure", "method"], values="median_signed_err_s").round(2)

## Figures

In [ ]:
for f in sorted(Path(OUT_DIR).glob("filters_*.png")):
    print(f.name); display(Image(filename=str(f)))